In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv('dataSP25.csv')

In [3]:
df.head()

,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365
0,1,Clinton Hill Apartment Single Bedroom ! Pratt ~~,30610258,Santiago,Brooklyn,Clinton Hill,40.69591,-73.96284,Private room,58,2,1,9/23/2018,0.10,1,0
1,2,Andy Phillips,6180052,Andrian,Brooklyn,Fort Greene,40.68769,-73.97490,Private room,99,5,4,9/29/2017,0.17,1,250
2,3,"Stylish, Comfortable, and Affordable in Brooklyn",5957027,Zachary,Brooklyn,Crown Heights,40.67667,-73.91750,Entire home/apt,180,2,126,6/14/2019,3.74,1,74
3,4,15 min train ride to Times Sq and US OPEN,101602599,M,Queens,Woodside,40.74340,-73.90588,Entire home/apt,50,1,95,6/24/2019,5.63,2,0
4,5,Sunny 2 bdrm Park Slope Apt w/ Yard 3min to Su...,106168581,Jason,Brooklyn,South Slope,40.66600,-73.99020,Entire home/apt,210,4,88,7/4/2019,4.41,1,20


In [4]:
df.isna().sum()

id                                   0
name                                 9
host_id                              0
host_name                           10
neighbourhood_group                  0
neighbourhood                        0
latitude                             0
longitude                            0
room_type                            0
price                                0
minimum_nights                       0
number_of_reviews                    0
last_review                       5633
reviews_per_month                 5633
calculated_host_listings_count       0
availability_365                     0
dtype: int64

In [5]:
df = df.drop(['name', 'last_review', 'reviews_per_month', 'host_name'], axis=1)

In [6]:
X = df.drop('price', axis=1)
y = df['price']

In [7]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [8]:
X_train.describe()

,id,host_id,latitude,longitude,minimum_nights,number_of_reviews,calculated_host_listings_count,availability_365
count,21903.000000,2.190300e+04,21903.000000,21903.000000,21903.000000,21903.000000,21903.000000,21903.000000
mean,13695.239602,6.768948e+07,40.728976,-73.952116,6.972835,23.466238,7.151395,112.957220
std,7906.066391,7.878280e+07,0.054602,0.046153,19.613125,44.800251,32.892064,131.697264
min,1.000000,2.438000e+03,40.507080,-74.242850,1.000000,0.000000,1.000000,0.000000
25%,6846.500000,7.870446e+06,40.690040,-73.983010,1.000000,1.000000,1.000000,0.000000
50%,13724.000000,3.055530e+07,40.723150,-73.955670,3.000000,5.000000,1.000000,45.000000
75%,20544.500000,1.074344e+08,40.763130,-73.936180,5.000000,24.000000,2.000000,229.000000
max,27378.000000,2.743115e+08,40.913060,-73.717950,999.000000,607.000000,327.000000,365.000000


In [9]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

In [10]:
X_train.columns

Index(['id', 'host_id', 'neighbourhood_group', 'neighbourhood', 'latitude',
       'longitude', 'room_type', 'minimum_nights', 'number_of_reviews',
       'calculated_host_listings_count', 'availability_365'],
      dtype='object')

In [11]:
scaler = StandardScaler()
cols = [
    'id', 'host_id', 'latitude', 'longitude',
    'minimum_nights', 'number_of_reviews',
    'calculated_host_listings_count', 'availability_365'
]
X_train[cols] = scaler.fit_transform(X_train[cols])
X_test[cols] = scaler.transform(X_test[cols])

In [12]:
oe = OrdinalEncoder(
    categories=[
        ['Entire home/apt', 'Private room', 'Shared room']
    ]
)

X_train['room_type'] = oe.fit_transform(X_train[['room_type']])
X_test['room_type'] = oe.transform(X_test[['room_type']])

In [13]:
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

X_train[ohe.get_feature_names_out()] = ohe.fit_transform(
    X_train[['neighbourhood_group', 'neighbourhood']]
)
X_test[ohe.get_feature_names_out()] = ohe.transform(
    X_test[['neighbourhood_group', 'neighbourhood']]
)


X_train = X_train.drop(['neighbourhood_group', 'neighbourhood'], axis=1)
X_test = X_test.drop(['neighbourhood_group', 'neighbourhood'], axis=1)

In [14]:
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_squared_error

In [15]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso

In [16]:
# for alpha in [0.001, 0.01, 0.1, 1, 10, 100]:
#     model = Lasso(alpha=alpha)
#     scores = cross_val_score(
#         model, X_train, y_train, cv=5, scoring="neg_root_mean_squared_error"
#     )
#     rmse = -np.mean(scores)
#     print(rmse)

In [17]:
model = LinearRegression()
scores = cross_val_score(
    model, X_train, y_train, cv=5, scoring="neg_root_mean_squared_error"
)
rmse = -np.mean(scores)
rmse

np.float64(211.1962288503279)

In [18]:
from sklearn.tree import DecisionTreeRegressor
model = DecisionTreeRegressor()
scores = cross_val_score(
    model, X_train, y_train, cv=5, scoring="neg_root_mean_squared_error"
)
rmse = -np.mean(scores)
rmse

np.float64(286.6097090142563)

In [19]:
from sklearn.neighbors import KNeighborsRegressor

In [36]:
for k in [1, 5, 9, 13, 17, 21, 25, 29]:
    model = KNeighborsRegressor(n_neighbors=k)
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring="neg_root_mean_squared_error")
    rmse = -np.mean(scores)
    print(rmse)

270.4855968855412
222.30197318815652
215.08947355224797
213.8040399281074
212.22840060386503
211.55446016244505
211.34879541102882
211.12820819503003


In [37]:
model = KNeighborsRegressor(n_neighbors=29)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
rmse

np.float64(161.05836059790073)

In [68]:
res = [None, float("inf")]
for alpha in [0.001, 0.01, 0.1, 1, 10, 100]:
    model = Ridge(alpha=alpha)
    scores = cross_val_score(
        model, X_train, y_train, cv=5, scoring="neg_root_mean_squared_error"
    )
    rmse = -np.mean(scores)
    if rmse < res[1]:
        res[0] = alpha
        res[1] = rmse
    print(rmse)
model = Ridge(alpha=res[0])
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(rmse)

211.19596391696342
211.19360392376507
211.17217739768859
211.0608809958053
210.8264563492668
210.77623622761075
161.3673855443769


In [39]:
# from sklearn.svm import SVC

In [40]:
# for C in [0.01, 0.1, 1, 10, 100]:
#     model = SVC(C=C)
#     scores = cross_val_score(
#         model, X_train, y_train, cv=5, scoring="neg_root_mean_squared_error"
#     )
#     rmse = -np.mean(scores)
#     print(rmse)

In [41]:
comp = pd.read_csv('compSP25.csv')

In [42]:
comp.isna().sum()

id                                   0
name                                 0
host_id                              0
host_name                            4
neighbourhood_group                  0
neighbourhood                        0
latitude                             0
longitude                            0
room_type                            0
minimum_nights                       0
number_of_reviews                    0
last_review                       1375
reviews_per_month                 1375
calculated_host_listings_count       0
availability_365                     0
dtype: int64

In [43]:
comp = comp.drop(['name', 'last_review', 'reviews_per_month', 'host_name'], axis=1)

In [44]:
comp.isna().sum()

id                                0
host_id                           0
neighbourhood_group               0
neighbourhood                     0
latitude                          0
longitude                         0
room_type                         0
minimum_nights                    0
number_of_reviews                 0
calculated_host_listings_count    0
availability_365                  0
dtype: int64

In [45]:
comp['room_type'] = oe.transform(comp[['room_type']])

In [46]:
comp[ohe.get_feature_names_out()] = ohe.transform(
    comp[['neighbourhood_group', 'neighbourhood']]
)
comp = comp.drop(['neighbourhood_group', 'neighbourhood'], axis=1)

In [47]:
final_pred = model.predict(comp)

In [48]:
final_pred

array([5.31322415e+06, 6.69124193e+08, 1.39242583e+09, ...,
       1.05899782e+09, 1.86301287e+08, 3.64257674e+07])

In [49]:
final_pred.shape

(6847,)

In [50]:
pd.DataFrame({
    'id': range(1, len(final_pred) + 1),
    'price': final_pred
}).to_csv('res.csv', index=False)

In [51]:
from sklearn.model_selection import GridSearchCV

In [66]:
model = Ridge()
param_grid = {
    'alpha': [0.001, 0.01, 0.1, 1, 10, 100]
}

grid = GridSearchCV(model, param_grid, cv=5, scoring="neg_root_mean_squared_error")

grid.fit(X_train, y_train)
y_pred = grid.predict(X_test)

print(grid.best_params_)
print(grid.best_score_)
print('final rmse:', np.sqrt(mean_squared_error(y_test, y_pred)))

{'alpha': 100}
-210.77623622761075
final rmse: 161.3673855443769
